<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks%20/DeiTTiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Land Cover Regression – DeiT Tiny Experiments

This notebook implements a multimodal regression framework that combines image features with textual descriptions to predict land cover distributions.

The experiments are designed to evaluate the contribution of textual information to visual models and the effect of different caption types listed in the following:
  - Text-only captions
  - Hybrid captions
  - Vision-generated captions

The impact of different lightweight text encoders, (MiniLM,DistilBERT, RemoteCLIP) are also observed.

The image backbone used in this notebook is Deit Tiny, which is fine-tuned during training. Text encoders are kept frozen, and features are combined using a gated fusion mechanism.

All experiments are run under a consistent setup (same split, seed, training configuration) to ensure fair comparison.

## Environment Setup

This section installs the required libraries for the experiments.
These packages provide the core functionality for building and training the multimodal model.

In [1]:
!pip install -q timm transformers accelerate wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00


## Data Loading and Setup

The dataset is loaded from Google Drive and extracted locally. This setup ensures that images and their corresponding textual descriptions can be accessed efficiently during training.

In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms

import wandb
import open_clip
from huggingface_hub import hf_hub_download

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
os.makedirs("/content/drive/MyDrive/DI725_term_project_phase3_predictions", exist_ok=True)

In [5]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [6]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
LOCAL_CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
LOCAL_IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

CSV_PATH = LOCAL_CSV_PATH
IMAGE_DIR = LOCAL_IMAGE_DIR

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


## Experiment Configuration

This cell defines the main experimental settings used throughout the notebook, including sample size, image size, batch size, number of epochs, learning rate, weight decay, early stopping patience, and random seed.

The target variables are the seven classes: Tree, Shrub, Grass, Crop, Built-up, Barren, and Water. The text_scale parameter controls the strength of the textual contribution during gated fusion.

In [7]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 224,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "deit_tiny_patch16_224",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 0.7
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

## Reproducibility Setup

This cell fixes the random seed across Python, NumPy, and PyTorch to improve reproducibility. Using the same seed helps ensure that sampling, data splitting, and model training are comparable across experiments.

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

## Data Loading and Text Cleaning

This section loads the caption and target data from the CSV file and prepares the text columns used in the multimodal experiments.

A cleaning function removes percentages, standalone numbers, and leakage related phrases from captions. This step prevents the model from directly using numerical target information contained in the text, while preserving semantic descriptions.

In [9]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_3520/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_3520/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_3520/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [10]:
df["hybrid_gemma3_4b_clean"].head(10)

,hybrid_gemma3_4b_clean
0,the image depicts a landscape dominated by ext...
1,the image depicts a largely arid landscape dom...
2,the image depicts a landscape dominated by ext...
3,the image depicts a valley dominated by dense ...
4,the image depicts a coastal area dominated by ...
5,the image depicts a landscape dominated by cul...
6,the image depicts a landscape dominated by a r...
7,the image depicts a hilly landscape dominated ...
8,the image depicts a landscape dominated by ext...
9,the image depicts a landscape dominated by ext...


In [11]:
df["hybrid_gemma3_4b_clean"].str.contains(
    r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
    regex=True,
    case=False
).sum()

/tmp/ipykernel_3520/1445695331.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["hybrid_gemma3_4b_clean"].str.contains(


np.int64(0)

In [12]:
df["text_qwen3_4b_clean"].str.contains(
    r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
    regex=True,
    case=False
).sum()

/tmp/ipykernel_3520/895902091.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["text_qwen3_4b_clean"].str.contains(


np.int64(0)

## Stratified Sampling and Data Split

This section prepares the dataset for training by first filtering out rows with missing values in required columns (images, targets, and cleaned captions).

A dominant class is computed for each sample based on the maximum target value across land-cover classes. This is used to perform stratified sampling, ensuring that the class distribution is preserved.

## Train / Validation / Test Split

The sampled dataset is split into 80% training, 10% validation, 10% test

Stratification is applied based on the dominant class to maintain consistent class distribution across splits. This ensures fair and reliable evaluation of model performance.

Indices are reset after splitting to avoid indexing issues during data loading.

In [13]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

# If available rows <= sample_size, use all rows
if len(df) <= CONFIG["sample_size"]:
    sample_df = df.sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
else:
    sample_df, _ = train_test_split(
        df,
        train_size=CONFIG["sample_size"],
        stratify=df["dominant_class"],
        random_state=CONFIG["seed"]
    )
    sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))

sample_df = sample_df.reset_index(drop=True)

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Sample size: 10000
Train: 8000
Val: 1000
Test: 1000


## Target Distribution Check

To verify that stratification is effective, this section computes summary statistics (mean and standard deviation) for each target class across the train, validation, and test splits.

This check ensures that the distributions of land cover classes remain consistent across splits, preventing bias in training or evaluation.

In [14]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,8000,28.8445,35.170320,0.828875,4.064469,45.367375,33.938682,17.95475,30.544088,1.139625,5.456130,4.192875,11.363640,1.672,9.244490
1,val,1000,28.8610,35.001154,0.818000,3.981297,45.064000,34.437681,18.15700,30.993789,1.190000,4.985053,4.181000,11.631231,1.729,9.926837
2,test,1000,28.1980,34.791741,0.941000,4.581103,45.318000,34.315075,18.14000,30.799334,1.095000,4.951615,4.508000,12.302898,1.800,9.461793


In [15]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470375,0.470,0.470
Tree,0.303750,0.304,0.303
Crop,0.180250,0.181,0.180
Barren,0.019250,0.019,0.020
Water,0.017750,0.018,0.018
Built-up,0.005875,0.006,0.006
Shrub,0.002750,0.002,0.003


## Image Transformations

This section defines preprocessing pipelines for training and evaluation.

Training transformations include resizing, random horizontal and vertical flips for data augmentation, and normalization using ImageNet statistics.

Evaluation transformations apply resizing and normalization only, ensuring consistent input without augmentation during validation and testing.

In [16]:
IMG_SIZE = CONFIG["img_size"]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Multimodal Dataset

This dataset class prepares inputs for multimodal training. For each sample the image is loaded and transformed and target values are extracted as a multi-output regression vector.
If a text column is provided, captions are tokenized using a pretrained tokenizer

The dataset returns, image tensor, target vector, tokenized text inputs. This structure enables joint processing of image and text features in the model.

In [17]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=128):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [18]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

## Image Only Baseline Model

This model serves as a baseline using only visual information.

A pretrained vision backbone (DeiT Tiny) is used to extract image features, which are then passed through a regression head to predict land-cover targets.

The regression head consists of layer normalization, fully connected layers,ReLU activation and dropout for regularization

This baseline allows comparison against multimodal models to measure the contribution of textual information. The backbone is initialized with pretrained weights and fine-tuned during training.

In [19]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=7):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

## Multimodal Model (Image + Text Fusion)

This model extends the image-only baseline by incorporating textual information through a gated fusion mechanism.

The architecture consists of:
- A pretrained vision backbone (Deit Tiny) for image feature extraction
- A pretrained text encoder (MiniLM / DistilBERT/ RemoteCLIP) for caption representation
- A projection layer to map text features into the image feature space

The text encoder is kept frozen during training to reduce computational cost and ensure stable representations.

To combine modalities, a gating mechanism is used. Image and text features are concatenated and passed through a learnable gate. The gate controls how much textual information contributes to the final representation. Moreover, a scaling factor further adjusts the strength of text features.

The final fused representation is computed as:

    fused = image_features + scale × gate × text_features

This design allows the model to adaptively use textual information depending on its relevance.

In [20]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=7,
        text_scale=0.7
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

In [21]:
class RemoteCLIPTextEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32"):
        super().__init__()

        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

        ckpt_path = hf_hub_download(
            repo_id="chendelong/RemoteCLIP",
            filename="RemoteCLIP-ViT-B-32.pt"
        )

        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]

        cleaned = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "").replace("model.", "")
            cleaned[k] = v

        self.model.load_state_dict(cleaned, strict=False)

        print("RemoteCLIP weights loaded")

        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, texts):
        device = next(self.model.parameters()).device
        tokens = self.tokenizer(list(texts)).to(device)

        with torch.no_grad():
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)

        return features

In [22]:
class DeitTinyRemoteCLIPFusion(nn.Module):
    def __init__(
        self,
        image_model_name,
        num_outputs=7,
        text_scale=0.7
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_encoder = RemoteCLIPTextEncoder(model_name="ViT-B-32")

        with torch.no_grad():
            dummy_text = ["satellite image of land cover"]
            dummy_features = self.text_encoder(dummy_text)
            text_dim = dummy_features.shape[1]

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image, texts):
        image_features = self.image_backbone(image)

        text_features = self.text_encoder(texts)
        text_features = text_features.to(image_features.device)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused = image_features + self.text_scale * gate * text_features

        return self.regressor(fused)

## Evaluation Functions

This section defines evaluation procedures for both image only and multimodal models.

Mean Absolute Error (MAE) is used as the primary evaluation metric. Root Mean Squared Error (RMSE) is also computed for additional analysis. Furthermore, Class-wise MAE is calculated to analyze performance across land-cover categories

These metrics provide both overall and class-level performance insights.

In [23]:
@torch.no_grad()
def evaluate_image_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

In [24]:
@torch.no_grad()
def evaluate_text_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        input_ids = batch["input_ids"].to(config["device"])
        attention_mask = batch["attention_mask"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images, input_ids, attention_mask)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

In [25]:
@torch.no_grad()
def evaluate_remoteclip_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        texts = batch["text"]
        targets = batch["target"].to(config["device"])

        preds = model(images, texts)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

## Training: Image-Only Model

This function defines the training loop for the image-only baseline. The loss function is Mean Absolute Error (L1 loss). The AdamW optimizer with weight decay is used.

Training progress is logged using Weights & Biases (W&B).

In [26]:
def train_image_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_image_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_mae

## Training: Multimodal Model

This function extends the training process to multimodal inputs.

Differences from image only training is that the image and tokenized text inputs are used. Moreover, the multimodal forward pass combines features using gated fusion.

The same training configuration is maintained. This ensures a fair comparison between image only and multimodal models.

In [27]:
def train_text_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            input_ids = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images, input_ids, attention_mask)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_text_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_mae

In [28]:
def train_remoteclip_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            texts = batch["text"]
            targets = batch["target"].to(config["device"])

            preds = model(images, texts)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_remoteclip_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_mae

In [29]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiment: Image Only Baseline

This function runs the image only experiment using the DeiTTiny backbone.

Metrics (MAE, RMSE, and class-wise MAE) are logged using Weights & Biases (W&B).

This experiment serves as the baseline for comparing multimodal configurations.

In [30]:
def run_deittiny_image_only(save_predictions=False):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "deittiny_image_only"
    run_config["fusion"] = "none"
    run_config["text_column"] = "none"
    run_config["text_model"] = "none"

    wandb.init(
        project="DI725-Project-landcover-regression_phase3_experiments",
        name="deittiny_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds = LandCoverDataset(val_df, IMAGE_DIR, transform=eval_tfms)
    test_ds = LandCoverDataset(test_df, IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_image_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_image_model(model, test_loader, CONFIG)

    if save_predictions:
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(
            f"/content/drive/MyDrive/DI725_term_project_phase3_predictions/predictions_{CONFIG['model_name']}_image_only.csv",
            index=False
        )

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
        "experiment": "deittiny_image_only",
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

## Experiment: Multimodal (Image + Text)

This function runs multimodal experiments by combining image features with textual information.

Different caption sources and text encoders (MiniLM, DistilBERT) are utilized

For each configuration the model is initialized with a gated fusion mechanism, text contribution is controlled via a scaling factor and the same training and evaluation pipeline is applied

Results are logged to W&B, enabling comparison across different text sources and encoders. This setup allows systematic evaluation of how textual information impacts model performance.

In [31]:
def run_deittiny_transformer_text(text_col, text_model, text_scale=0.7, wandb_project="DI725-Project-landcover-regression_phase3_experiments",save_predictions=False):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "deittiny_transformer_text"
    run_config["text_column"] = text_col
    run_config["text_model"] = text_model
    run_config["text_scale"] = text_scale
    run_config["fusion"] = "gated_frozen_scaled"

    run_name = f"deittiny_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(
        project=wandb_project,
        name=run_name,
        config=run_config,
        reinit=True
    )

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_text_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_text_model(model, test_loader, CONFIG)

    if save_predictions:
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(
            f"/content/drive/MyDrive/DI725_term_project_phase3_predictions/predictions_{CONFIG['model_name']}_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}.csv",
            index=False
        )

    wandb.log({
        "test_loss":    test_loss,
        "test_mae":     test_mae,
        "test_rmse":    test_rmse,
        "best_val_mae": best_val_mae
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })

    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  text_model,
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

In [32]:
def run_deittiny_remoteclip_text(text_col, text_scale=0.7, wandb_project="DI725-Project-landcover-regression_phase3_experiments",save_predictions=False):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "deittiny_remoteclip_text"
    run_config["text_column"] = text_col
    run_config["text_model"]  = "RemoteCLIP ViT-B-32"
    run_config["text_scale"]  = text_scale
    run_config["fusion"]      = "gated_frozen_scaled"

    run_name = f"deittiny_remoteclip_{text_col}_scale_{text_scale}"

    wandb.init(
        project=wandb_project,
        name=run_name,
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverRawTextDataset(train_df, IMAGE_DIR, transform=train_tfms, text_col=text_col)
    val_ds   = LandCoverRawTextDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  text_col=text_col)
    test_ds  = LandCoverRawTextDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = DeitTinyRemoteCLIPFusion(
        image_model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_remoteclip_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_remoteclip_model(model, test_loader, CONFIG)

    if save_predictions:
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(
            f"/content/drive/MyDrive/DI725_term_project_phase3_predictions/predictions_{CONFIG['model_name']}_{text_col}_remoteclip_scale_{text_scale}.csv",
            index=False
        )

    wandb.log({
        "test_loss":    test_loss,
        "test_mae":     test_mae,
        "test_rmse":    test_rmse,
        "best_val_mae": best_val_mae
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })

    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

# Running Experiments

In [33]:
results = []

## Image Only Experiment

In [34]:
results.append(run_deittiny_image_only(save_predictions=True))
pd.DataFrame(results)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

Epoch 1/15 | Train Loss: 12.6353 | Val MAE: 11.0626 | Val RMSE: 25.0771
Epoch 2/15 | Train Loss: 9.8554 | Val MAE: 8.2232 | Val RMSE: 18.8613
Epoch 3/15 | Train Loss: 7.0606 | Val MAE: 5.6273 | Val RMSE: 13.3355
Epoch 4/15 | Train Loss: 5.3907 | Val MAE: 4.0547 | Val RMSE: 10.4898
Epoch 5/15 | Train Loss: 4.4808 | Val MAE: 3.5631 | Val RMSE: 9.2507
Epoch 6/15 | Train Loss: 3.9943 | Val MAE: 3.3257 | Val RMSE: 8.6745
Epoch 7/15 | Train Loss: 3.7106 | Val MAE: 3.3019 | Val RMSE: 8.4900
Epoch 8/15 | Train Loss: 3.5665 | Val MAE: 3.2516 | Val RMSE: 8.6451
Epoch 9/15 | Train Loss: 3.3682 | Val MAE: 3.1635 | Val RMSE: 8.2145
Epoch 10/15 | Train Loss: 3.1912 | Val MAE: 2.8543 | Val RMSE: 8.0790
Epoch 11/15 | Train Loss: 3.0970 | Val MAE: 2.8209 | Val RMSE: 7.5329
Epoch 12/15 | Train Loss: 2.9776 | Val MAE: 2.8420 | Val RMSE: 7.8109
Epoch 13/15 | Train Loss: 2.8424 | Val MAE: 2.7407 | Val RMSE: 7.7273
Epoch 14/15 | Train Loss: 2.7272 | Val MAE: 2.7794 | Val RMSE: 7.4405
Epoch 15/15 | Train Los

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▃▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▆▃▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▆▃▂▂▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,2.83214
test_mae,2.83214


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,deittiny_image_only,2.83214,7.564394,2.865544,0.94838,6.899146,4.944613,0.837229,2.449494,0.880577


## MiniLM Experiments

In [35]:
TEXT_COLS_CLEAN = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

In [36]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_transformer_text(
            text_col=text_col,
            text_model="sentence-transformers/all-MiniLM-L6-v2",
            text_scale=0.7,
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.7322 | Val MAE: 11.2451 | Val RMSE: 25.1769
Epoch 2/15 | Train Loss: 10.0018 | Val MAE: 8.4626 | Val RMSE: 18.8616
Epoch 3/15 | Train Loss: 7.3820 | Val MAE: 5.7253 | Val RMSE: 13.6117
Epoch 4/15 | Train Loss: 5.5690 | Val MAE: 4.6039 | Val RMSE: 11.1786
Epoch 5/15 | Train Loss: 4.6013 | Val MAE: 3.7492 | Val RMSE: 9.4081
Epoch 6/15 | Train Loss: 4.0252 | Val MAE: 3.3597 | Val RMSE: 8.6717
Epoch 7/15 | Train Loss: 3.7349 | Val MAE: 3.0623 | Val RMSE: 8.5153
Epoch 8/15 | Train Loss: 3.5610 | Val MAE: 3.0907 | Val RMSE: 8.2359
Epoch 9/15 | Train Loss: 3.3955 | Val MAE: 3.1701 | Val RMSE: 8.2614
Epoch 10/15 | Train Loss: 3.2535 | Val MAE: 2.9050 | Val RMSE: 8.0787
Epoch 11/15 | Train Loss: 3.1260 | Val MAE: 2.8655 | Val RMSE: 8.0661
Epoch 12/15 | Train Loss: 3.0155 | Val MAE: 2.9325 | Val RMSE: 8.0277
Epoch 13/15 | Train Loss: 2.8854 | Val MAE: 2.9396 | Val RMSE: 7.9589
Epoch 14/15 | Train Loss: 2.7924 | Val MAE: 2.8406 | Val RMSE: 7.5441
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▃▃▂▁▁▁▁▁▁▁▁▁▁
val_mae,█▆▃▃▂▁▁▁▁▁▁▁▁▁▁
val_rmse,█▆▃▃▂▂▁▁▁▁▁▁▁▁▁
best_val_mae,2.77794
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,2.341449,5.927781,2.891298,0.948127,5.009615,3.318976,0.736020,2.511364,0.974743,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.309746,"[2.8912983, 0.94812703, 5.009615, 3.3189762, 0..."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,2.828061,7.148192,3.150759,0.956698,6.043099,4.423930,1.005468,2.918850,1.297625,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.789830,"[3.1507587, 0.9566975, 6.0430994, 4.4239297, 1..."
0,deittiny_image_only,2.832140,7.564394,2.865544,0.948380,6.899146,4.944613,0.837229,2.449494,0.880577,NaN,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,2.868724,7.440432,3.202398,0.946900,6.652613,4.632592,0.991511,2.531847,1.123207,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.777937,"[3.202398, 0.9468999, 6.6526127, 4.6325917, 0...."


## Remote CLIP Experiments

In [37]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_remoteclip_text(
            text_col=text_col,
            text_scale=0.7,
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7832 | Val MAE: 11.3090 | Val RMSE: 25.4169
Epoch 2/15 | Train Loss: 10.1866 | Val MAE: 8.4676 | Val RMSE: 19.2576
Epoch 3/15 | Train Loss: 7.4607 | Val MAE: 5.9549 | Val RMSE: 14.0673
Epoch 4/15 | Train Loss: 5.6551 | Val MAE: 4.3643 | Val RMSE: 10.9390
Epoch 5/15 | Train Loss: 4.6109 | Val MAE: 3.7198 | Val RMSE: 9.2013
Epoch 6/15 | Train Loss: 4.0601 | Val MAE: 3.1444 | Val RMSE: 8.4540
Epoch 7/15 | Train Loss: 3.7204 | Val MAE: 3.0665 | Val RMSE: 8.4974
Epoch 8/15 | Train Loss: 3.5238 | Val MAE: 2.9578 | Val RMSE: 8.0944
Epoch 9/15 | Train Loss: 3.3108 | Val MAE: 2.8253 | Val RMSE: 7.6329
Epoch 10/15 | Train Loss: 3.2181 | Val MAE: 2.8431 | Val RMSE: 7.5087
Epoch 11/15 | Train Loss: 3.0687 | Val MAE: 2.8720 | Val RMSE: 7.6973
Epoch 12/15 | Train Loss: 2.9065 | Val MAE: 2.5847 | Val RMSE: 7.1264
Epoch 13/15 | Train Loss: 2.8320 | Val MAE: 2.6378 | Val RMSE: 7.0628
Epoch 14/15 | Train Loss: 2.6907 | Val MAE: 2.6181 | Val RMSE: 6.9

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▂▂▁▁▁▁▁▁▁▁▁▁
val_mae,█▆▄▂▂▁▁▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
best_val_mae,2.56899
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7720 | Val MAE: 11.2661 | Val RMSE: 25.4107
Epoch 2/15 | Train Loss: 10.1366 | Val MAE: 8.5264 | Val RMSE: 19.2442
Epoch 3/15 | Train Loss: 7.4482 | Val MAE: 5.7586 | Val RMSE: 13.8078
Epoch 4/15 | Train Loss: 5.5994 | Val MAE: 4.3041 | Val RMSE: 10.6834
Epoch 5/15 | Train Loss: 4.6116 | Val MAE: 4.0873 | Val RMSE: 10.0679
Epoch 6/15 | Train Loss: 4.0429 | Val MAE: 3.4283 | Val RMSE: 9.0230
Epoch 7/15 | Train Loss: 3.6961 | Val MAE: 3.0372 | Val RMSE: 8.1562
Epoch 8/15 | Train Loss: 3.4180 | Val MAE: 2.9064 | Val RMSE: 7.7408
Epoch 9/15 | Train Loss: 3.2847 | Val MAE: 3.0564 | Val RMSE: 8.2667
Epoch 10/15 | Train Loss: 3.1353 | Val MAE: 2.7331 | Val RMSE: 7.3155
Epoch 11/15 | Train Loss: 2.8887 | Val MAE: 2.8990 | Val RMSE: 7.2571
Epoch 12/15 | Train Loss: 2.7990 | Val MAE: 2.7990 | Val RMSE: 7.1178
Epoch 13/15 | Train Loss: 2.6863 | Val MAE: 2.4571 | Val RMSE: 6.5678
Epoch 14/15 | Train Loss: 2.5665 | Val MAE: 2.6579 | Val RMSE: 6.

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▂▁▂▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▂▁▂▁▁▁▁▁▁
val_rmse,█▆▄▃▃▂▂▂▂▂▁▁▁▁▁
best_val_mae,2.29584
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7776 | Val MAE: 11.3406 | Val RMSE: 25.4323
Epoch 2/15 | Train Loss: 10.1254 | Val MAE: 8.4573 | Val RMSE: 19.1934
Epoch 3/15 | Train Loss: 7.4606 | Val MAE: 5.8290 | Val RMSE: 13.8381
Epoch 4/15 | Train Loss: 5.6894 | Val MAE: 4.6727 | Val RMSE: 11.1455
Epoch 5/15 | Train Loss: 4.7205 | Val MAE: 3.7513 | Val RMSE: 9.3107
Epoch 6/15 | Train Loss: 4.1469 | Val MAE: 3.2780 | Val RMSE: 8.4175
Epoch 7/15 | Train Loss: 3.7621 | Val MAE: 3.3203 | Val RMSE: 8.9427
Epoch 8/15 | Train Loss: 3.5839 | Val MAE: 3.0173 | Val RMSE: 8.2567
Epoch 9/15 | Train Loss: 3.3836 | Val MAE: 3.0906 | Val RMSE: 8.4224
Epoch 10/15 | Train Loss: 3.2529 | Val MAE: 3.0122 | Val RMSE: 8.0721
Epoch 11/15 | Train Loss: 3.1888 | Val MAE: 2.8105 | Val RMSE: 7.5481
Epoch 12/15 | Train Loss: 3.0325 | Val MAE: 2.9265 | Val RMSE: 7.6516
Epoch 13/15 | Train Loss: 2.9183 | Val MAE: 2.6796 | Val RMSE: 7.3035
Epoch 14/15 | Train Loss: 2.8104 | Val MAE: 2.6602 | Val RMSE: 7.2

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
best_val_mae,2.53338
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,2.341449,5.927781,2.891298,0.948127,5.009615,3.318976,0.736020,2.511364,0.974743,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.309746,"[2.8912983, 0.94812703, 5.009615, 3.3189762, 0..."
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,2.419510,6.214271,2.801850,0.962707,5.266859,3.721049,0.921092,2.443281,0.819735,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.295844,"[2.8018498, 0.96270657, 5.2668586, 3.721049, 0..."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,2.628688,7.000269,2.844006,0.952575,5.936004,4.211403,0.900177,2.725391,0.831261,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.568986,"[2.844006, 0.9525751, 5.9360037, 4.211403, 0.9..."
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,2.681823,7.173601,2.866653,0.966799,5.992042,4.466695,0.993174,2.504917,0.982476,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.533384,"[2.8666532, 0.96679926, 5.992042, 4.4666953, 0..."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,2.828061,7.148192,3.150759,0.956698,6.043099,4.423930,1.005468,2.918850,1.297625,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.789830,"[3.1507587, 0.9566975, 6.0430994, 4.4239297, 1..."
0,deittiny_image_only,2.832140,7.564394,2.865544,0.948380,6.899146,4.944613,0.837229,2.449494,0.880577,NaN,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,2.868724,7.440432,3.202398,0.946900,6.652613,4.632592,0.991511,2.531847,1.123207,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.777937,"[3.202398, 0.9468999, 6.6526127, 4.6325917, 0...."


## DistilBERT Experiments

In [38]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_transformer_text(
            text_col=text_col,
            text_model="distilbert-base-uncased",
            text_scale=0.7,
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8811 | Val MAE: 11.4521 | Val RMSE: 25.7660
Epoch 2/15 | Train Loss: 10.2612 | Val MAE: 8.5642 | Val RMSE: 19.3847
Epoch 3/15 | Train Loss: 7.5725 | Val MAE: 6.4107 | Val RMSE: 14.3266
Epoch 4/15 | Train Loss: 5.6971 | Val MAE: 5.0241 | Val RMSE: 12.0456
Epoch 5/15 | Train Loss: 4.6057 | Val MAE: 3.5723 | Val RMSE: 9.2790
Epoch 6/15 | Train Loss: 4.1007 | Val MAE: 3.6132 | Val RMSE: 8.9176
Epoch 7/15 | Train Loss: 3.7928 | Val MAE: 3.3996 | Val RMSE: 9.1315
Epoch 8/15 | Train Loss: 3.6854 | Val MAE: 3.2205 | Val RMSE: 8.4571
Epoch 9/15 | Train Loss: 3.4825 | Val MAE: 2.9264 | Val RMSE: 8.2255
Epoch 10/15 | Train Loss: 3.3353 | Val MAE: 2.8663 | Val RMSE: 8.0197
Epoch 11/15 | Train Loss: 3.1904 | Val MAE: 3.0900 | Val RMSE: 8.1244
Epoch 12/15 | Train Loss: 3.0083 | Val MAE: 2.8352 | Val RMSE: 7.7978
Epoch 13/15 | Train Loss: 2.9136 | Val MAE: 3.0201 | Val RMSE: 8.0126
Epoch 14/15 | Train Loss: 2.8245 | Val MAE: 2.7019 | Val RMSE: 7.3692
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,2.62209
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
8,deittiny_text_qwen3_4b_clean_distilbert-base-u...,2.303413,5.939941,2.698237,0.950501,5.114232,3.242676,0.718584,2.556535,0.843125,text_qwen3_4b_clean,distilbert-base-uncased,0.7,2.277080,"[2.6982374, 0.9505011, 5.1142316, 3.2426755, 0..."
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,2.341449,5.927781,2.891298,0.948127,5.009615,3.318976,0.736020,2.511364,0.974743,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.309746,"[2.8912983, 0.94812703, 5.009615, 3.3189762, 0..."
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,2.419510,6.214271,2.801850,0.962707,5.266859,3.721049,0.921092,2.443281,0.819735,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.295844,"[2.8018498, 0.96270657, 5.2668586, 3.721049, 0..."
7,deittiny_hybrid_gemma3_4b_clean_distilbert-bas...,2.578934,6.792444,2.781089,0.949794,6.049700,4.150191,0.715922,2.497933,0.907908,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.480501,"[2.7810893, 0.9497941, 6.0497003, 4.150191, 0...."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,2.628688,7.000269,2.844006,0.952575,5.936004,4.211403,0.900177,2.725391,0.831261,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.568986,"[2.844006, 0.9525751, 5.9360037, 4.211403, 0.9..."
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,2.681823,7.173601,2.866653,0.966799,5.992042,4.466695,0.993174,2.504917,0.982476,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.533384,"[2.8666532, 0.96679926, 5.992042, 4.4666953, 0..."
9,deittiny_vision_gemma3_4b_clean_distilbert-bas...,2.765518,7.325369,3.026879,0.965669,6.455020,4.612401,0.781367,2.621395,0.895891,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.622086,"[3.0268788, 0.9656691, 6.4550204, 4.6124015, 0..."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,2.828061,7.148192,3.150759,0.956698,6.043099,4.423930,1.005468,2.918850,1.297625,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.789830,"[3.1507587, 0.9566975, 6.0430994, 4.4239297, 1..."
0,deittiny_image_only,2.832140,7.564394,2.865544,0.948380,6.899146,4.944613,0.837229,2.449494,0.880577,NaN,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,2.868724,7.440432,3.202398,0.946900,6.652613,4.632592,0.991511,2.531847,1.123207,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.777937,"[3.202398, 0.9468999, 6.6526127, 4.6325917, 0...."


## Deit Tiny Experiment Results Summary

In [39]:
results_df = pd.DataFrame(results).sort_values("test_mae")
results_df

,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
8,deittiny_text_qwen3_4b_clean_distilbert-base-u...,2.303413,5.939941,2.698237,0.950501,5.114232,3.242676,0.718584,2.556535,0.843125,text_qwen3_4b_clean,distilbert-base-uncased,0.7,2.277080,"[2.6982374, 0.9505011, 5.1142316, 3.2426755, 0..."
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,2.341449,5.927781,2.891298,0.948127,5.009615,3.318976,0.736020,2.511364,0.974743,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.309746,"[2.8912983, 0.94812703, 5.009615, 3.3189762, 0..."
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,2.419510,6.214271,2.801850,0.962707,5.266859,3.721049,0.921092,2.443281,0.819735,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.295844,"[2.8018498, 0.96270657, 5.2668586, 3.721049, 0..."
7,deittiny_hybrid_gemma3_4b_clean_distilbert-bas...,2.578934,6.792444,2.781089,0.949794,6.049700,4.150191,0.715922,2.497933,0.907908,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.480501,"[2.7810893, 0.9497941, 6.0497003, 4.150191, 0...."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,2.628688,7.000269,2.844006,0.952575,5.936004,4.211403,0.900177,2.725391,0.831261,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.568986,"[2.844006, 0.9525751, 5.9360037, 4.211403, 0.9..."
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,2.681823,7.173601,2.866653,0.966799,5.992042,4.466695,0.993174,2.504917,0.982476,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.533384,"[2.8666532, 0.96679926, 5.992042, 4.4666953, 0..."
9,deittiny_vision_gemma3_4b_clean_distilbert-bas...,2.765518,7.325369,3.026879,0.965669,6.455020,4.612401,0.781367,2.621395,0.895891,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.622086,"[3.0268788, 0.9656691, 6.4550204, 4.6124015, 0..."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,2.828061,7.148192,3.150759,0.956698,6.043099,4.423930,1.005468,2.918850,1.297625,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.789830,"[3.1507587, 0.9566975, 6.0430994, 4.4239297, 1..."
0,deittiny_image_only,2.832140,7.564394,2.865544,0.948380,6.899146,4.944613,0.837229,2.449494,0.880577,NaN,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,2.868724,7.440432,3.202398,0.946900,6.652613,4.632592,0.991511,2.531847,1.123207,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.777937,"[3.202398, 0.9468999, 6.6526127, 4.6325917, 0...."


Experiments are conducted using three caption types (text only, hybrid, and vision based) and three text encoders (MiniLM, RemoteCLIP and DistilBERT). A fixed text scaling factor is applied to control the contribution of textual features in the fusion process.

## Text Scale Experiments

The best caption × encoder combination for DeiT-Tiny based on
validation MAE at scale 0.7 is selected.

In [40]:
# Filter out image-only row — selection is over multimodal configs only
multimodal_results = [r for r in results if r.get("text_model") not in [None, "none"]]
results_df_multimodal = pd.DataFrame(multimodal_results)

best_config = results_df_multimodal.loc[results_df_multimodal["val_mae"].idxmin()]

print("Best configuration for DeiT-Tiny (by validation MAE)")
print(f"  Text column : {best_config['text_column']}")
print(f"  Text model  : {best_config['text_model']}")
print(f"  Val MAE     : {best_config['val_mae']:.4f}")
print(f"  Test MAE    : {best_config['test_mae']:.4f}")

BEST_TEXT_COL   = best_config["text_column"]
BEST_TEXT_MODEL = best_config["text_model"]
IS_REMOTECLIP   = (BEST_TEXT_MODEL == "RemoteCLIP ViT-B-32")

Best configuration for DeiT-Tiny (by validation MAE)
  Text column : text_qwen3_4b_clean
  Text model  : distilbert-base-uncased
  Val MAE     : 2.2771
  Test MAE    : 2.3034


## Text Scale Ablation

Using the best configuration selected above, we sweep text_scale across
[0.3, 0.5, 0.7, 1.0, 1.5].

In [41]:
SCALE_WANDB_PROJECT = "DI725-Project-landcover-regression_phase3_text_scale_exp"
TEXT_SCALES = [0.3, 0.5, 0.7, 1.0, 1.5]

scale_results = []

for scale in TEXT_SCALES:
    print(f"\nRunning scale={scale} | col={BEST_TEXT_COL} | model={BEST_TEXT_MODEL}")

    if IS_REMOTECLIP:
        result = run_deittiny_remoteclip_text(
            text_col=BEST_TEXT_COL,
            text_scale=scale,
            wandb_project=SCALE_WANDB_PROJECT,
            save_predictions=True
        )
    else:
        result = run_deittiny_transformer_text(
            text_col=BEST_TEXT_COL,
            text_model=BEST_TEXT_MODEL,
            text_scale=scale,
            wandb_project=SCALE_WANDB_PROJECT,
            save_predictions=True
        )

    scale_results.append(result)

scale_df = pd.DataFrame(scale_results).sort_values("text_scale").reset_index(drop=True)
scale_df


Running scale=0.3 | col=text_qwen3_4b_clean | model=distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8876 | Val MAE: 11.4256 | Val RMSE: 25.7783
Epoch 2/15 | Train Loss: 10.2335 | Val MAE: 8.5696 | Val RMSE: 19.3282
Epoch 3/15 | Train Loss: 7.5814 | Val MAE: 6.5560 | Val RMSE: 14.6154
Epoch 4/15 | Train Loss: 5.7884 | Val MAE: 4.4913 | Val RMSE: 10.7449
Epoch 5/15 | Train Loss: 4.6410 | Val MAE: 3.9072 | Val RMSE: 9.5294
Epoch 6/15 | Train Loss: 3.9545 | Val MAE: 3.5417 | Val RMSE: 8.7879
Epoch 7/15 | Train Loss: 3.7095 | Val MAE: 2.9874 | Val RMSE: 8.0855
Epoch 8/15 | Train Loss: 3.4765 | Val MAE: 3.0211 | Val RMSE: 7.9316
Epoch 9/15 | Train Loss: 3.2923 | Val MAE: 2.8885 | Val RMSE: 7.8924
Epoch 10/15 | Train Loss: 3.1429 | Val MAE: 3.0888 | Val RMSE: 8.1652
Epoch 11/15 | Train Loss: 3.0302 | Val MAE: 2.6074 | Val RMSE: 7.1739
Epoch 12/15 | Train Loss: 2.8562 | Val MAE: 2.7469 | Val RMSE: 7.2334
Epoch 13/15 | Train Loss: 2.6971 | Val MAE: 2.4753 | Val RMSE: 6.8070
Epoch 14/15 | Train Loss: 2.6387 | Val MAE: 2.4193 | Val RMSE: 6.4945
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▂▂▁▂▁▁▁▁▁
val_mae,█▆▄▃▂▂▂▂▁▂▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▂▂▂▁▁▁▁▁
best_val_mae,2.29452
epoch,15


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8608 | Val MAE: 11.4016 | Val RMSE: 25.6666
Epoch 2/15 | Train Loss: 10.2238 | Val MAE: 8.6013 | Val RMSE: 19.4585
Epoch 3/15 | Train Loss: 7.3929 | Val MAE: 5.9454 | Val RMSE: 13.5879
Epoch 4/15 | Train Loss: 5.4823 | Val MAE: 4.1601 | Val RMSE: 10.0660
Epoch 5/15 | Train Loss: 4.3552 | Val MAE: 3.2728 | Val RMSE: 8.4670
Epoch 6/15 | Train Loss: 3.7744 | Val MAE: 3.0024 | Val RMSE: 7.9683
Epoch 7/15 | Train Loss: 3.5348 | Val MAE: 2.9209 | Val RMSE: 7.8502
Epoch 8/15 | Train Loss: 3.3351 | Val MAE: 2.8118 | Val RMSE: 7.4918
Epoch 9/15 | Train Loss: 3.1664 | Val MAE: 2.6928 | Val RMSE: 7.3069
Epoch 10/15 | Train Loss: 3.0600 | Val MAE: 2.5741 | Val RMSE: 7.0447
Epoch 11/15 | Train Loss: 2.8736 | Val MAE: 2.7877 | Val RMSE: 7.0554
Epoch 12/15 | Train Loss: 2.7469 | Val MAE: 2.4939 | Val RMSE: 6.6244
Epoch 13/15 | Train Loss: 2.6281 | Val MAE: 2.3669 | Val RMSE: 6.2783
Epoch 14/15 | Train Loss: 2.5387 | Val MAE: 2.1991 | Val RMSE: 5.9185
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▂▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▆▄▂▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▆▄▂▂▂▂▂▂▁▁▁▁▁▁
best_val_mae,2.19915
epoch,15


,experiment,text_column,text_model,text_scale,val_mae,test_mae,test_rmse,class_mae,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.3,2.294522,2.365563,6.051821,"[2.79998, 0.95169896, 5.303817, 3.4045641, 0.7...",2.799980,0.951699,5.303817,3.404564,0.700866,2.386842,1.011173
1,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.5,2.315072,2.352068,5.895546,"[3.187412, 0.9575019, 5.117239, 3.1968622, 0.7...",3.187412,0.957502,5.117239,3.196862,0.760536,2.404439,0.840484
2,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.7,2.285602,2.353750,5.992035,"[2.780585, 0.94700503, 5.178568, 3.3429813, 0....",2.780585,0.947005,5.178568,3.342981,0.773511,2.456344,0.997255
3,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,1.0,2.360471,2.394877,6.120272,"[2.9396784, 0.9483323, 5.152187, 3.4440837, 0....",2.939678,0.948332,5.152187,3.444084,0.758710,2.690071,0.831077
4,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,1.5,2.199147,2.269543,5.786277,"[2.6981554, 0.9559624, 4.9661508, 3.2492092, 0...",2.698155,0.955962,4.966151,3.249209,0.711713,2.537077,0.768535


## Text Scale Ablation — Overall MAE Summary

In [42]:
image_only_row = next(r for r in results if r.get("text_model") in [None, "none"])
baseline_mae   = image_only_row["test_mae"]

scale_df["delta_vs_baseline"] = (baseline_mae - scale_df["test_mae"]).round(4)
scale_df["pct_improvement"]   = ((scale_df["delta_vs_baseline"] / baseline_mae) * 100).round(2)

print(f"Image-only baseline test MAE : {baseline_mae:.4f}\n")
print(f"Best config : {BEST_TEXT_COL}  +  {BEST_TEXT_MODEL}\n")

display_cols = ["text_scale", "val_mae", "test_mae", "test_rmse", "delta_vs_baseline", "pct_improvement"]
scale_df[display_cols]

Image-only baseline test MAE : 2.8321

Best config : text_qwen3_4b_clean  +  distilbert-base-uncased



,text_scale,val_mae,test_mae,test_rmse,delta_vs_baseline,pct_improvement
0,0.3,2.294522,2.365563,6.051821,0.4666,16.48
1,0.5,2.315072,2.352068,5.895546,0.4801,16.95
2,0.7,2.285602,2.353750,5.992035,0.4784,16.89
3,1.0,2.360471,2.394877,6.120272,0.4373,15.44
4,1.5,2.199147,2.269543,5.786277,0.5626,19.86


## Text Scale Ablation — Class-wise MAE

In [43]:
class_mae_cols = [f"mae_{cls}" for cls in TARGET_COLS]
class_scale_df = scale_df[["text_scale"] + class_mae_cols].copy()
class_scale_df.columns = ["text_scale"] + TARGET_COLS

baseline_class_mae = np.array([image_only_row[f"mae_{cls}"] for cls in TARGET_COLS])

delta_rows = []
for _, row in class_scale_df.iterrows():
    row_mae = np.array([row[cls] for cls in TARGET_COLS])
    delta_rows.append(
        {"text_scale": row["text_scale"],
         **{cls: round(baseline_class_mae[i] - row_mae[i], 4) for i, cls in enumerate(TARGET_COLS)}}
    )

delta_class_df = pd.DataFrame(delta_rows)

print("Class-wise MAE per text scale")
print(class_scale_df.to_string(index=False))

print("\nClass-wise improvement vs image-only baseline (positive = better)")
print(delta_class_df.to_string(index=False))

Class-wise MAE per text scale
 text_scale     Tree    Shrub    Grass     Crop  Built-up   Barren    Water
        0.3 2.799980 0.951699 5.303817 3.404564  0.700866 2.386842 1.011173
        0.5 3.187412 0.957502 5.117239 3.196862  0.760536 2.404439 0.840484
        0.7 2.780585 0.947005 5.178568 3.342981  0.773511 2.456344 0.997255
        1.0 2.939678 0.948332 5.152187 3.444084  0.758710 2.690071 0.831077
        1.5 2.698155 0.955962 4.966151 3.249209  0.711713 2.537077 0.768535

Class-wise improvement vs image-only baseline (positive = better)
 text_scale    Tree   Shrub  Grass   Crop  Built-up  Barren   Water
        0.3  0.0656 -0.0033 1.5953 1.5400    0.1364  0.0627 -0.1306
        0.5 -0.3219 -0.0091 1.7819 1.7478    0.0767  0.0451  0.0401
        0.7  0.0850  0.0014 1.7206 1.6016    0.0637 -0.0069 -0.1167
        1.0 -0.0741  0.0000 1.7470 1.5005    0.0785 -0.2406  0.0495
        1.5  0.1674 -0.0076 1.9330 1.6954    0.1255 -0.0876  0.1120


## Full Results Summary — All DeiT-Tiny Experiments

In [44]:
scale_result_experiments = {r["experiment"] for r in scale_results}

all_results_df = pd.DataFrame(results + scale_results).copy()

all_results_df["experiment_type"] = all_results_df.apply(
    lambda r: "image_only"  if pd.isna(r.get("text_model")) or r.get("text_model") in [None, "none"]
    else ("scale_sweep"     if r.get("experiment") in scale_result_experiments
    else  "multimodal_0.7"),
    axis=1
)

display_cols = [
    "experiment_type", "text_column", "text_model",
    "text_scale", "val_mae", "test_mae", "test_rmse"
]

all_results_df[display_cols].sort_values(
    ["experiment_type", "test_mae"]
).reset_index(drop=True)

,experiment_type,text_column,text_model,text_scale,val_mae,test_mae,test_rmse
0,image_only,NaN,NaN,NaN,NaN,2.832140,7.564394
1,multimodal_0.7,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.309746,2.341449,5.927781
2,multimodal_0.7,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.295844,2.419510,6.214271
3,multimodal_0.7,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.480501,2.578934,6.792444
4,multimodal_0.7,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.568986,2.628688,7.000269
5,multimodal_0.7,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.533384,2.681823,7.173601
6,multimodal_0.7,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.622086,2.765518,7.325369
7,multimodal_0.7,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.789830,2.828061,7.148192
8,multimodal_0.7,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.777937,2.868724,7.440432
9,scale_sweep,text_qwen3_4b_clean,distilbert-base-uncased,1.5,2.199147,2.269543,5.786277


In [45]:
# Best configuration overall — across all experiments including scale sweep
all_multimodal = all_results_df[all_results_df["experiment_type"] != "image_only"]
best_overall = all_multimodal.loc[all_multimodal["val_mae"].idxmin()]

print("Best overall configuration (by validation MAE)")
print(f"  Experiment type : {best_overall['experiment_type']}")
print(f"  Text column     : {best_overall['text_column']}")
print(f"  Text model      : {best_overall['text_model']}")
print(f"  Text scale      : {best_overall['text_scale']}")
print(f"  Val MAE         : {best_overall['val_mae']:.4f}")
print(f"  Test MAE        : {best_overall['test_mae']:.4f}")
print(f"  Test RMSE       : {best_overall['test_rmse']:.4f}")

Best overall configuration (by validation MAE)
  Experiment type : scale_sweep
  Text column     : text_qwen3_4b_clean
  Text model      : distilbert-base-uncased
  Text scale      : 1.5
  Val MAE         : 2.1991
  Test MAE        : 2.2695
  Test RMSE       : 5.7863


In [46]:
BACKBONE_NAME = "deittiny"
all_results_df.to_csv(
    f"/content/drive/MyDrive/DI725_term_project_phase3_predictions/all_results_{BACKBONE_NAME}.csv",
    index=False
)
print("Saved results to Drive")

Saved results to Drive
